# Let's try some machine learning!

## Create the Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML


class GlyphClassifier(nn.Module):
    def __init__(self, NUM_classes, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_classes)
        )
 
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the output
        x = self.classifier(x)
        return x

In [ ]:
NUM_classes = 100 #Number of classes can be set here

# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train", bins=NUM_classes)
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test', bins=NUM_classes)
# train_dataset = ML.GlyphDataset('data/simple-star-L.zip', split = "train", num_classes=NUM_classes)
# test_dataset = ML.GlyphDataset('data/simple-star-L.zip', split = 'test', num_classes=NUM_classes)

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=2)


In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = GlyphClassifier(NUM_classes, resolution=(128, 128)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
losses = []

for epoch in range(num_epochs):
    model.train()
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {losses[-1]:.4f}")


In [ ]:
ML.plot_training_loss(losses)

In [ ]:
all_true_labels = []
all_predicted_labels = []

model.eval()
correct_predictions = 0
total_samples = 0
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted_labels == labels).sum().item()
        all_true_labels.extend(labels.cpu().numpy())
        all_predicted_labels.extend(predicted_labels.cpu().numpy())



accuracy = 100 * correct_predictions / total_samples
print(f'\n--- Evaluation Results ---')
print(f'Total Test Samples: {total_samples}')
print(f'Correct Predictions: {correct_predictions}')
print(f'Accuracy on the test set: {accuracy:.2f}%')
print(f'--------------------------')



class_names = [f'Class {i}' for i in range(NUM_classes)]

print("\nGenerating Confusion Matrix...")
ML.plot_confusion_matrix(
    y_true=all_true_labels,
    y_pred=all_predicted_labels,
    classes=class_names,
    normalize=False
)
plt.show() 



In [ ]:
ML.show_incorrect_predictions(model, test_loader, num_classes=NUM_classes, max_display=10, device=device)

## **More Complicated Training** 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import src.machine_learning as ML
import torch

# === Load your saved experiment results ===
df = pd.read_pickle('experiment_results.pkl')

# Select experiment: 10 classes at 32x32 resolution
exp = df[(df['num_classes'] == 20) & (df['resolution'] == (128, 128))].iloc[0]

# === Plot Confusion Matrix ===
ML.plot_confusion_matrix(
    y_true=exp['true_labels'],
    y_pred=exp['predicted_labels'],
    classes=[f'Class {i}' for i in range(exp['num_classes'])],
    normalize=False
)
plt.show()

In [ ]:

#  resnet-50, pre-trained on ImageNet
